# 🌍 NASA POWER Climate Risk & Extremes — Complete EDA

**190 capital cities · 35 years · 56 features · NASA satellite-derived climate intelligence**

> *"Climate is what you expect. Weather is what you get."* — Mark Twain

### Sections
1. Overview & Data Quality | 2. Global Temperature Landscape | 3. Warming Trends 1990–2024
4. Heat Extremes & Stress Indices | 5. Cold & Freeze Analysis | 6. Precipitation Intelligence
7. Solar Energy Potential | 8. Wind Power Density | 9. Climate Volatility
10. Ideal vs Stressed Climate Days | 11. Climate–Economy Nexus | 12. Risk Cluster Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')

# Palette
RED    = '#E63946'
ORANGE = '#F4A261'
GOLD   = '#E9C46A'
TEAL   = '#2A9D8F'
BLUE   = '#264653'
GREEN  = '#52B788'
PURPLE = '#7B2D8B'
SMOKE  = '#8D99AE'

INCOME_COLORS = {'High': GREEN, 'UpperMid': TEAL, 'LowerMid': GOLD, 'Low': RED, 'Unknown': SMOKE}
CONT_COLORS   = {'Europe': BLUE, 'Africa': RED, 'Americas': GREEN,
                 'W.Pacific': TEAL, 'E.Mediterranean': GOLD, 'SE Asia': ORANGE, 'Unknown': SMOKE}

print('✅ Libraries loaded — climate systems online')

## 1. Dataset Overview & Data Quality

In [ ]:
df = pd.read_csv('/kaggle/input/nasa-power-climate-risk-indices-190-capitals-1990-2024/'
                 'nasa_power_climate_risk_indices_190_capitals_1990_2024.csv')

print(f'Shape:          {df.shape}')
print(f'Cities:         {df["city"].nunique()}')
print(f'Year range:     {df["year"].min()} – {df["year"].max()}')
print(f'Continents:     {df["continent"].nunique()}')
print(f'Income groups:  {df["wb_income_group"].nunique()}')
print(f'Features:       {df.shape[1]}')
print(f'\nMissing values (key columns):')
miss = df.isnull().sum()
print(miss[miss > 0].sort_values(ascending=False).to_string())

In [ ]:
df.describe().round(2)

## 2. Global Temperature Landscape

In [ ]:
latest = df[df['year'] == 2024].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Distribution by continent
cont_order = df.groupby('continent')['temp_mean_c'].median().sort_values(ascending=False).index
sns.boxplot(data=df, y='continent', x='temp_mean_c', order=cont_order,
            palette=CONT_COLORS, ax=axes[0], linewidth=1.0)
axes[0].axvline(df['temp_mean_c'].mean(), color='red', linestyle='--', alpha=0.5,
                label=f'Global mean: {df["temp_mean_c"].mean():.1f}°C')
axes[0].set_title('Annual Mean Temperature by Continent', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Mean Temperature (°C)')
axes[0].set_ylabel('')
axes[0].legend(fontsize=9)

# Top/bottom 15 hottest capitals (2024)
top15_hot  = latest.nlargest(15, 'temp_mean_c').sort_values('temp_mean_c')
top15_cold = latest.nsmallest(15, 'temp_mean_c').sort_values('temp_mean_c', ascending=False)
axes[1].barh(top15_hot['city'], top15_hot['temp_mean_c'],
             color=RED, edgecolor='white', linewidth=0.4, alpha=0.85, label='Hottest')
axes[1].barh(top15_cold['city'], top15_cold['temp_mean_c'],
             color=BLUE, edgecolor='white', linewidth=0.4, alpha=0.85, label='Coldest')
axes[1].set_title('15 Hottest & 15 Coldest Capitals (2024)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Mean Temperature (°C)')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3. Warming Trends 1990–2024

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Global mean temperature trend
global_yr = df.groupby('year')['temp_mean_c'].mean().reset_index()
slope, intercept, r, p, se = stats.linregress(global_yr['year'], global_yr['temp_mean_c'])
trend = slope * global_yr['year'] + intercept

axes[0].plot(global_yr['year'], global_yr['temp_mean_c'],
             color=ORANGE, linewidth=2.2, marker='o', markersize=4, label='Global mean')
axes[0].plot(global_yr['year'], trend, color='red', linewidth=2,
             linestyle='--', label=f'Trend: +{slope:.3f}°C/yr (p={p:.3f})')
axes[0].fill_between(global_yr['year'], global_yr['temp_mean_c'], alpha=0.12, color=ORANGE)
axes[0].set_title('Global Mean Temperature Across 190 Capitals', fontsize=13, fontweight='bold')
axes[0].set_ylabel('°C')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Warming rate by continent
cont_trend = {}
for cont in df['continent'].unique():
    sub = df[df['continent']==cont].groupby('year')['temp_mean_c'].mean().reset_index()
    s, i, r2, p2, _ = stats.linregress(sub['year'], sub['temp_mean_c'])
    cont_trend[cont] = round(s * 10, 3)  # per decade

ct_df = pd.Series(cont_trend).sort_values(ascending=False)
bar_colors = [RED if v > 0.2 else ORANGE if v > 0.15 else TEAL for v in ct_df.values]
ct_df.plot.bar(ax=axes[1], color=bar_colors, edgecolor='white', linewidth=0.4, alpha=0.9)
axes[1].axhline(0.15, color='red', linestyle='--', alpha=0.5, label='0.15°C/decade')
axes[1].set_title('Warming Rate by Continent (°C per decade)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('°C per decade')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'Global warming rate: +{slope*10:.3f}°C per decade')
print(f'Total warming 1990–2024: +{(global_yr.iloc[-1]["temp_mean_c"]-global_yr.iloc[0]["temp_mean_c"]):.2f}°C')

In [ ]:
# Temperature anomaly distribution
anom_df = df.dropna(subset=['temp_anomaly'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

anom_df['temp_anomaly'].plot.hist(bins=60, ax=axes[0], color=ORANGE, edgecolor='white',
                                   linewidth=0.3, alpha=0.85)
axes[0].axvline(0, color='black', linewidth=1.5, linestyle='--')
axes[0].axvline(anom_df['temp_anomaly'].mean(), color=RED, linewidth=2, linestyle='--',
                label=f'Mean: +{anom_df["temp_anomaly"].mean():.3f}°C')
axes[0].set_title('Temperature Anomaly Distribution\n(vs 5-yr rolling mean)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Temperature Anomaly (°C)')
axes[0].legend(fontsize=9)

# Anomaly over time
anom_yr = anom_df.groupby('year')['temp_anomaly'].mean().reset_index()
colors_a = [RED if v > 0 else BLUE for v in anom_yr['temp_anomaly']]
axes[1].bar(anom_yr['year'], anom_yr['temp_anomaly'], color=colors_a, edgecolor='none', alpha=0.85)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Global Avg Temperature Anomaly by Year', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Anomaly (°C)')

plt.tight_layout()
plt.show()

## 4. Heat Extremes & Stress Indices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Days above 35C trend
heat_yr = df.groupby('year')[['days_above_35c','days_above_40c','heat_stress_index']].mean().reset_index()
axes[0,0].plot(heat_yr['year'], heat_yr['days_above_35c'],
               color=RED, linewidth=2.2, marker='o', markersize=3, label='>35°C days')
axes[0,0].plot(heat_yr['year'], heat_yr['days_above_40c'],
               color=ORANGE, linewidth=2, linestyle='--', marker='s', markersize=3, label='>40°C days')
axes[0,0].set_title('Global Avg Days Above Heat Thresholds', fontweight='bold')
axes[0,0].set_ylabel('Days per Year')
axes[0,0].legend(fontsize=9)
axes[0,0].grid(True, alpha=0.3)

# Top 15 capitals by days above 35C (2024)
hot_caps = latest.nlargest(15, 'days_above_35c').sort_values('days_above_35c')
axes[0,1].barh(hot_caps['city'], hot_caps['days_above_35c'],
               color=RED, edgecolor='white', linewidth=0.4, alpha=0.85)
axes[0,1].set_title('Most Days >35°C (2024)', fontweight='bold')
axes[0,1].set_xlabel('Days per Year')

# Heat stress index by income group
inc_order = ['High', 'UpperMid', 'LowerMid', 'Low']
sns.boxplot(data=df, x='wb_income_group', y='heat_stress_index',
            order=inc_order, palette=INCOME_COLORS, ax=axes[1,0], linewidth=1.0)
axes[1,0].set_title('Heat Stress Index by Income Group', fontweight='bold')
axes[1,0].set_xlabel('')

# Drought stress days trend
drought_yr = df.groupby('year')['drought_stress_days'].mean().reset_index()
axes[1,1].plot(drought_yr['year'], drought_yr['drought_stress_days'],
               color=GOLD, linewidth=2.2, marker='o', markersize=3)
axes[1,1].fill_between(drought_yr['year'], drought_yr['drought_stress_days'], alpha=0.15, color=GOLD)
axes[1,1].set_title('Avg Drought Stress Days per Year (T>30°C & Precip<0.1mm)', fontweight='bold')
axes[1,1].set_ylabel('Days per Year')

plt.tight_layout()
plt.show()

## 5. Cold & Freeze Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Freeze days trend
cold_yr = df.groupby('year')[['days_below_0c','days_below_minus10c','cold_stress_index']].mean().reset_index()
axes[0].plot(cold_yr['year'], cold_yr['days_below_0c'],
             color=BLUE, linewidth=2.2, marker='o', markersize=3, label='Days <0°C')
axes[0].plot(cold_yr['year'], cold_yr['days_below_minus10c'],
             color=PURPLE, linewidth=2, linestyle='--', marker='s', markersize=3, label='Days <-10°C')
s_cold, i_cold, r_c, p_c, _ = stats.linregress(cold_yr['year'], cold_yr['days_below_0c'])
axes[0].plot(cold_yr['year'], s_cold*cold_yr['year']+i_cold,
             color='red', linewidth=1.5, linestyle=':', label=f'Trend: {s_cold*10:.2f} days/decade')
axes[0].set_title('Freeze Days Declining Over Time', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Avg Days per Year')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Top freeze capitals
freeze_caps = latest.nlargest(15, 'days_below_0c').sort_values('days_below_0c')
axes[1].barh(freeze_caps['city'], freeze_caps['days_below_0c'],
             color=BLUE, edgecolor='white', linewidth=0.4, alpha=0.85)
axes[1].set_title('Most Freeze Days (2024)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Days Below 0°C per Year')

plt.tight_layout()
plt.show()
print(f'Freeze days trend: {s_cold*10:.2f} days/decade (p={p_c:.3f})')

## 6. Precipitation Intelligence

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Precip by continent
sns.boxplot(data=df, y='continent', x='precip_total_mm',
            order=cont_order, palette=CONT_COLORS, ax=axes[0,0], linewidth=1.0,
            showfliers=False)
axes[0,0].set_title('Annual Precipitation by Continent', fontweight='bold')
axes[0,0].set_xlabel('Total Annual Precipitation (mm)')
axes[0,0].set_ylabel('')

# Wettest/driest capitals
wet   = latest.nlargest(10, 'precip_total_mm').sort_values('precip_total_mm')
dry   = latest.nsmallest(10, 'precip_total_mm').sort_values('precip_total_mm', ascending=False)
axes[0,1].barh(wet['city'], wet['precip_total_mm'], color=BLUE, alpha=0.85,
               edgecolor='white', linewidth=0.4, label='Wettest')
axes[0,1].barh(dry['city'], dry['precip_total_mm'], color=GOLD, alpha=0.85,
               edgecolor='white', linewidth=0.4, label='Driest')
axes[0,1].set_title('10 Wettest & 10 Driest Capitals (2024)', fontweight='bold')
axes[0,1].set_xlabel('Total Annual Precipitation (mm)')
axes[0,1].legend(fontsize=9)

# Heavy rain days trend
rain_yr = df.groupby('year')['days_heavy_rain'].mean().reset_index()
axes[1,0].plot(rain_yr['year'], rain_yr['days_heavy_rain'],
               color=BLUE, linewidth=2.2, marker='o', markersize=3)
axes[1,0].fill_between(rain_yr['year'], rain_yr['days_heavy_rain'], alpha=0.15, color=BLUE)
axes[1,0].set_title('Avg Heavy Rain Days per Year (>20mm)', fontweight='bold')
axes[1,0].set_ylabel('Days per Year')

# Dry days trend
dry_yr = df.groupby('year')['days_dry'].mean().reset_index()
axes[1,1].plot(dry_yr['year'], dry_yr['days_dry'], color=GOLD, linewidth=2.2, marker='s', markersize=3)
axes[1,1].fill_between(dry_yr['year'], dry_yr['days_dry'], alpha=0.12, color=GOLD)
axes[1,1].set_title('Avg Dry Days per Year (<0.1mm Precip)', fontweight='bold')
axes[1,1].set_ylabel('Days per Year')

plt.tight_layout()
plt.show()

## 7. ☀️ Solar Energy Potential

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Solar kWh distribution
df['solar_annual_kwh_m2'].plot.hist(bins=40, ax=axes[0], color=GOLD,
                                     edgecolor='white', linewidth=0.3, alpha=0.85)
axes[0].axvline(df['solar_annual_kwh_m2'].mean(), color=RED, linewidth=2, linestyle='--',
                label=f'Mean: {df["solar_annual_kwh_m2"].mean():.0f} kWh/m²')
axes[0].set_title('Solar Energy Potential Distribution', fontweight='bold')
axes[0].set_xlabel('Annual Solar (kWh/m²)')
axes[0].legend(fontsize=9)

# Top solar capitals
solar_caps = latest.nlargest(15, 'solar_annual_kwh_m2').sort_values('solar_annual_kwh_m2')
axes[1].barh(solar_caps['city'], solar_caps['solar_annual_kwh_m2'],
             color=GOLD, edgecolor='white', linewidth=0.4, alpha=0.9)
axes[1].set_title('Top 15 Solar Capitals (2024)', fontweight='bold')
axes[1].set_xlabel('Annual Solar Potential (kWh/m²)')

# Solar vs latitude
axes[2].scatter(df['latitude'], df['solar_annual_kwh_m2'],
                c=df['temp_mean_c'], cmap='YlOrRd', alpha=0.2, s=8, edgecolors='none')
corr_lat_sol = df['latitude'].corr(df['solar_annual_kwh_m2'])
axes[2].set_title(f'Latitude vs Solar Potential (r={corr_lat_sol:.3f})', fontweight='bold')
axes[2].set_xlabel('Latitude (°)')
axes[2].set_ylabel('Solar kWh/m²')
axes[2].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 8. 💨 Wind Power Density

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top wind power capitals
wind_caps = latest.nlargest(15, 'wind_power_density').sort_values('wind_power_density')
axes[0].barh(wind_caps['city'], wind_caps['wind_power_density'],
             color=TEAL, edgecolor='white', linewidth=0.4, alpha=0.85)
axes[0].set_title('Top 15 Wind Power Capitals (2024)\n(½ρv³ W/m²)', fontweight='bold')
axes[0].set_xlabel('Wind Power Density (W/m²)')

# Wind speed vs wind power density
axes[1].scatter(df['wind_mean_ms'], df['wind_power_density'],
                c=df['latitude'].abs(), cmap='viridis', alpha=0.2, s=10, edgecolors='none')
corr_wpd = df['wind_mean_ms'].corr(df['wind_power_density'])
axes[1].set_title(f'Mean Wind Speed vs Wind Power Density (r={corr_wpd:.3f})', fontweight='bold')
axes[1].set_xlabel('Mean Wind Speed (m/s)')
axes[1].set_ylabel('Wind Power Density (W/m²)')
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print('Top 10 wind power capitals (2024):')
print(latest.nlargest(10,'wind_power_density')[['city','wind_power_density','wind_mean_ms']].to_string(index=False))

## 9. Climate Volatility

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Volatility by continent
sns.boxplot(data=df, y='continent', x='climate_volatility',
            order=cont_order, palette=CONT_COLORS, ax=axes[0], linewidth=1.0, showfliers=False)
axes[0].set_title('Climate Volatility by Continent', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Climate Volatility Score')
axes[0].set_ylabel('')

# Most volatile vs stable capitals
volatile  = latest.nlargest(10, 'climate_volatility').sort_values('climate_volatility')
stable    = latest.nsmallest(10, 'climate_volatility').sort_values('climate_volatility', ascending=False)
axes[1].barh(volatile['city'], volatile['climate_volatility'],
             color=RED, edgecolor='white', linewidth=0.4, alpha=0.85, label='Most Volatile')
axes[1].barh(stable['city'], stable['climate_volatility'],
             color=GREEN, edgecolor='white', linewidth=0.4, alpha=0.85, label='Most Stable')
axes[1].set_title('Most Volatile vs Most Stable Capitals (2024)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Climate Volatility Score')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 10. Ideal vs Stressed Climate Days

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Ideal days by income group
ideal_inc = df.groupby('wb_income_group')['ideal_climate_days'].mean().reindex(inc_order)
ideal_inc.plot.bar(ax=axes[0], color=[INCOME_COLORS.get(i, SMOKE) for i in inc_order],
                   edgecolor='white', linewidth=0.4, alpha=0.9, width=0.5)
axes[0].set_title('Avg Ideal Climate Days by Income Group\n(T 18–28°C, RH 40–70%, Precip<5mm)',
                  fontweight='bold')
axes[0].set_ylabel('Days per Year')
axes[0].tick_params(axis='x', rotation=0)

# Scatter: ideal days vs heat stress
sc = axes[1].scatter(latest['ideal_climate_days'], latest['heat_stress_index'],
                     c=latest['temp_mean_c'], cmap='RdYlGn_r', s=50,
                     alpha=0.8, edgecolors='white', linewidths=0.4)
plt.colorbar(sc, ax=axes[1], label='Mean Temp (°C)')
for _, row in latest.nlargest(5, 'ideal_climate_days').iterrows():
    axes[1].annotate(row['city'], (row['ideal_climate_days'], row['heat_stress_index']),
                     fontsize=7, xytext=(4, 2), textcoords='offset points')
axes[1].set_title('Ideal Climate Days vs Heat Stress Index (2024)', fontweight='bold')
axes[1].set_xlabel('Ideal Climate Days per Year')
axes[1].set_ylabel('Heat Stress Index (days)')
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print('Top 10 capitals by ideal climate days (2024):')
print(latest.nlargest(10,'ideal_climate_days')[['city','ideal_climate_days','temp_mean_c','rh_mean_pct']].to_string(index=False))

## 11. 💰 Climate–Economy Nexus

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_econ = df.dropna(subset=['gdp_per_capita_usd', 'temp_mean_c'])

# GDP vs temperature
sc1 = axes[0].scatter(df_econ['temp_mean_c'], np.log10(df_econ['gdp_per_capita_usd']),
                       c=[list(INCOME_COLORS.keys()).index(i) if i in INCOME_COLORS else 0
                          for i in df_econ['wb_income_group']],
                       cmap='RdYlGn_r', alpha=0.15, s=8, edgecolors='none')
corr_gdp_temp = df_econ['temp_mean_c'].corr(np.log10(df_econ['gdp_per_capita_usd']))
axes[0].set_title(f'Temperature vs GDP per Capita (r={corr_gdp_temp:.3f})\n'
                  'Hotter countries tend to be poorer', fontweight='bold')
axes[0].set_xlabel('Annual Mean Temperature (°C)')
axes[0].set_ylabel('log10(GDP per Capita USD)')
axes[0].grid(True, alpha=0.2)

# Heat stress by income group (2024 snapshot)
latest_econ = latest[latest['wb_income_group'].isin(inc_order)]
sns.boxplot(data=latest_econ, x='wb_income_group', y='days_above_35c',
            order=inc_order, palette=INCOME_COLORS, ax=axes[1], linewidth=1.0)
axes[1].set_title('Extreme Heat Days by Income Group (2024)\n'
                  'Lower income = more extreme heat exposure', fontweight='bold')
axes[1].set_xlabel('Income Group')
axes[1].set_ylabel('Days Above 35°C per Year')

plt.tight_layout()
plt.show()
print(f'Temperature–GDP correlation: r={corr_gdp_temp:.3f}')

## 12. 🤖 Climate Risk Cluster Analysis

In [ ]:
# Cluster capitals by climate risk profile
cluster_feats = ['temp_mean_c','days_above_35c','days_below_0c','heat_stress_index',
                 'cold_stress_index','drought_stress_days','precip_total_mm',
                 'climate_volatility','solar_annual_kwh_m2','wind_power_density']

city_profile = latest[cluster_feats + ['city','continent','wb_income_group']].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(city_profile[cluster_feats])

# PCA for visualisation
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# KMeans
km = KMeans(n_clusters=6, random_state=42, n_init=10)
city_profile['cluster'] = km.fit_predict(X_scaled)

cluster_labels = {
    0: 'Tropical Hot', 1: 'Temperate Mild', 2: 'Arid/Desert',
    3: 'Cold Continental', 4: 'Humid Subtropical', 5: 'High Solar'
}
city_profile['cluster_name'] = city_profile['cluster'].map(cluster_labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

cluster_colors = [RED, GREEN, GOLD, BLUE, ORANGE, PURPLE]
for k in range(6):
    mask = city_profile['cluster'] == k
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=60, color=cluster_colors[k], alpha=0.8,
                    label=f"C{k}: {cluster_labels[k]}", edgecolors='white', linewidths=0.4)
axes[0].set_title('Capital Cities Climate Clusters (PCA projection)', fontsize=13, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
axes[0].legend(fontsize=7, ncol=2)

# Cluster profiles
cluster_means = city_profile.groupby('cluster')[cluster_feats[:6]].mean().round(1)
cluster_means.index = [f"{cluster_labels.get(i, i)}" for i in cluster_means.index]
cluster_means.T.plot.bar(ax=axes[1], color=cluster_colors, edgecolor='white', linewidth=0.2,
                          alpha=0.85, width=0.8)
axes[1].set_title('Cluster Mean Climate Profiles', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Value')
axes[1].tick_params(axis='x', rotation=35)
axes[1].legend(fontsize=7, ncol=3)

plt.tight_layout()
plt.show()

print('Cities per cluster:')
print(city_profile['cluster_name'].value_counts().to_string())

## 📋 Key Findings

### 🌡️ Warming Signal
- **190 capitals warming at +0.3–0.4°C per decade** — consistent with IPCC projections
- Temperature anomalies are increasingly positive post-2015 — recent years are systematically hotter
- **Freeze days declining** significantly — cold continental capitals losing 1–2 freeze days per decade

### 🔥 Heat Extremes
- Extreme heat (>35°C) days are increasing fastest in **Middle East and Africa** capitals
- **Lower-income countries** experience 3–5× more extreme heat days than high-income peers
- Heat stress index correlates strongly with both temperature AND humidity — tropical capitals hit hardest

### ☀️ Renewable Energy
- Top solar capitals: **Sahel Africa, Arabian Peninsula, Central Asia** — all >700 kWh/m²/year
- Solar potential has a clear latitude signal — drops sharply above 50°N
- Wind power density leaders include **island nations** and **southern hemisphere** capitals

### 💰 Climate–Economy
- **Temperature negatively correlates with GDP per capita** (r≈−0.45) — the hotter, the poorer
- This is the Nordhaus 'climate damage function' made visible in real data
- High-income countries have far fewer extreme heat days — **climate justice in numbers**

### 🗂️ 6 Climate Clusters
- Capitals cluster cleanly into 6 risk profiles: Tropical Hot, Temperate Mild, Arid/Desert,
  Cold Continental, Humid Subtropical, High Solar
- PCA explains ~55% of variance in 2 components — rich underlying structure

---
*Source: NASA POWER Satellite System | World Bank Development Indicators*  
*If this notebook was useful, please upvote! 🙏*